# Offline run viewer

Latest frozen run under `runs/`. No lab needed.

In [ ]:
import json

import pandas as pd
from IPython.display import Markdown, display

from adsim.exercise import Run

EXER_RUNS_DIR = "runs"

run = Run.latest(EXER_RUNS_DIR, with_file="artifacts-suricata-eve.json")
run

In [ ]:
print(json.dumps(run.marker, indent=2))

In [ ]:
!ls -lah {run.dir}

## EVE

In [ ]:
DF = pd.json_normalize(json.loads((run.dir / "artifacts-suricata-eve.json").read_text()))
DF.groupby(["event_type"]).size()

In [ ]:
def uniq(s):
    vals = s.dropna().map(lambda v: str(int(v)) if isinstance(v, float) and v.is_integer() else str(v))
    return ", ".join(vals.loc[~vals.isin(["", "<empty>"])].unique())

META = [c for c in [
    "krb5.cname",
    "krb5.sname",
    "smb.share",
    "smb.filename",
    "fileinfo.filename",
] if c in DF.columns]

SEQUENCE = (
    DF
    .assign(**{"dcerpc.interfaces": lambda df: df["dcerpc.interfaces"].map(
        lambda l: ", ".join(i["uuid"] for i in l) if isinstance(l, list) else None)})
    .groupby("flow_id")
    .agg(
        first_seen=("timestamp", "min"),
        src=("src_ip", "first"),
        dest=("dest_ip", "first"),
        port=("dest_port", "first"),
        event_type=("event_type", uniq),
        records=("event_type", "count"),
        bytes_toserver=("flow.bytes_toserver", "max"),
        bytes_toclient=("flow.bytes_toclient", "max"),
        alerts=("alert.signature", uniq),
        **{c: (c, uniq) for c in META},
    )
    .assign(metadata=lambda df: df[META].apply(
        lambda r: "; ".join(f"{k}: {v}" for k, v in r.items() if v), axis=1))
    .drop(columns=META)
    .sort_values(by=["first_seen", "port"])
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(SEQUENCE)

## Findings

In [ ]:
findings_path = run.dir / "findings.md"

print(findings_path)

if findings_path.is_file():
    display(Markdown(findings_path.read_text().replace("$", r"\$")))
else:
    print("no findings.md in this run")